# 第8章：DPO 直接偏好优化

## 本章目标
- 理解 DPO 相比 RLHF 的优势（无需 Reward Model）
- 理解 DPO loss 的推导思路
- 使用 trl 的 DPOTrainer 完成 DPO 训练
- 对比 DPO vs RLHF 的效果和成本

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install torch transformers trl peft datasets accelerate bitsandbytes
else:
    print("本地环境运行，请确保已按 intro.md 配置好环境")

## DPO 速览

DPO (Direct Preference Optimization) 的核心洞察：**跳过 Reward Model，直接用偏好数据优化策略**。

RLHF 流程：SFT → Train Reward Model → PPO (需要 3 个模型)
DPO 流程：SFT → DPO (只需要 2 个模型：policy + reference)

DPO loss: L = -log σ(β · (log π(y_w)/π_ref(y_w) - log π(y_l)/π_ref(y_l)))

直觉：让模型增大 chosen 的概率，减小 rejected 的概率，同时用 reference model 做正则化。

参考：[DPO 论文](https://arxiv.org/abs/2305.18290)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import torch

model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map="auto"
)

ref_model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map="auto"
)

dataset = load_dataset("Anthropic/hh-rlhf", split="train[:3000]")
print(f"数据集大小: {len(dataset)}")
print("Policy model + Reference model 加载完成")

In [ ]:
from trl import DPOTrainer, DPOConfig
from peft import LoraConfig, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16, lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
)

training_args = DPOConfig(
    output_dir="./dpo_output",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    max_length=512,
    beta=0.1,
)

trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)
trainer.train()
print("DPO 训练完成")

In [ ]:
def dpo_loss(policy_chosen_logps, policy_rejected_logps,
             ref_chosen_logps, ref_rejected_logps, beta=0.1):
    """手动实现 DPO loss 用于理解"""
    chosen_rewards = beta * (policy_chosen_logps - ref_chosen_logps)
    rejected_rewards = beta * (policy_rejected_logps - ref_rejected_logps)
    loss = -torch.nn.functional.logsigmoid(chosen_rewards - rejected_rewards).mean()
    return loss

# 验证
policy_chosen = torch.tensor([-1.0, -2.0, -1.5])
policy_rejected = torch.tensor([-3.0, -2.5, -4.0])
ref_chosen = torch.tensor([-1.0, -2.0, -1.5])
ref_rejected = torch.tensor([-3.0, -2.5, -4.0])
loss = dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected)
print(f"DPO loss example: {loss.item():.4f}")
print()
print("关键理解：")
print("- chosen_rewards = β × (log π_chosen - log π_ref_chosen)")
print("- 如果 policy 比 reference 更倾向于 chosen → reward 为正 → loss 降低")
print("- 如果 policy 比 reference 更倾向于 rejected → reward 为负 → loss 升高")

## DPO vs RLHF 对比

| 维度 | RLHF (PPO) | DPO |
|------|-----------|-----|
| 需要的模型数 | 3 (policy + reward + ref) | 2 (policy + ref) |
| 训练复杂度 | 高（需要在线采样） | 低（离线，直接优化） |
| 显存需求 | 高 | 低 |
| 训练稳定性 | 较差（超参敏感） | 较好 |
| 理论保证 | 强（最优策略） | 强（闭式解） |
| 适用场景 | 有在线反馈循环 | 有大量偏好数据 |

**实践建议**：先用 DPO，如果效果不够再尝试 RLHF。DPO 的性价比通常更高。

参考：[DPO vs RLHF 分析](https://huggingface.co/blog/dpo-rlhf)

## 练习

1. 调整 `beta` (0.05, 0.1, 0.5)，观察训练 loss 的变化
2. 对比 DPO 和 PPO 在相同数据上的最终效果
3. 尝试不同的 LoRA rank，观察对 DPO 效果的影响

## 延伸阅读

- [DPO 论文](https://arxiv.org/abs/2305.18290)
- [trl DPOTrainer 文档](https://huggingface.co/docs/trl/dpo_trainer)
- [IPO: 一种 DPO 的替代方案](https://arxiv.org/abs/2310.12036)
- [RLHF vs DPO 实战对比](https://huggingface.co/blog/dpo-rlhf)